In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -q -U transformers datasets peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 22.0 MB/s eta 0:00:00


In [5]:
import torch
import json
import os

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel
)

SELECT MODEL

In [6]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("Model:", model_name)

Model: Qwen/Qwen2.5-0.5B-Instruct


LOAD TOKENIZER

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded
Vocabulary size: 151643


In [8]:
print("EOS token:", tokenizer.eos_token)
print("EOS ID:", tokenizer.eos_token_id)

print("PAD token:", tokenizer.pad_token)
print("PAD ID:", tokenizer.pad_token_id)

EOS token: <|im_end|>
EOS ID: 151645
PAD token: <|endoftext|>
PAD ID: 151643


In [9]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("PAD token:", tokenizer.pad_token)

PAD token: <|endoftext|>


BASE MODEL

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)

print("Model loaded on:", device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cuda


In [11]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

TEST ORIGINAL MODEL

In [12]:
def generate_response(model, tokenizer, prompt, max_new_tokens=100):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

TEST IT

BEFORE LORA RESULT

In [13]:
prompt = "Explain LoRA in simple words."

response = generate_response(
    model,
    tokenizer,
    prompt
)

print(response)

Explain LoRA in simple words. LoRA stands for "Learning Rate Optimization". It is a technique used in machine learning to adjust the learning rate of a neural network during training, which helps it converge faster and more efficiently.
The learning rate determines how quickly the model updates its weights based on gradients calculated from the loss function during each iteration of the training process. A high learning rate can lead to overshooting local minima, while a low one may result in slow convergence or even divergence.
By adjusting the learning rate, you are essentially


TRAINING DATASET

In [14]:
data = [

    {
        "instruction": "What is LoRA?",
        "response": "LoRA stands for Low-Rank Adaptation. It is a parameter-efficient fine-tuning technique that freezes the pretrained model and trains small low-rank matrices."
    },

    {
        "instruction": "What is RAG?",
        "response": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from an external knowledge source and provides that information to a language model before generating an answer."
    },

    {
        "instruction": "What is an LLM?",
        "response": "An LLM is a Large Language Model trained on large amounts of text to understand and generate human-like language."
    },

    {
        "instruction": "What is a Transformer?",
        "response": "A Transformer is a neural network architecture based on attention mechanisms. Modern language models use Transformer layers to process relationships between tokens."
    },

    {
        "instruction": "What is fine-tuning?",
        "response": "Fine-tuning means taking a pretrained model and training it further on a smaller task-specific dataset so that it performs better for a particular task."
    },

    {
        "instruction": "What is PEFT?",
        "response": "PEFT stands for Parameter-Efficient Fine-Tuning. It adapts pretrained models by training only a small number of additional or selected parameters instead of updating the entire model."
    },

    {
        "instruction": "What is an embedding?",
        "response": "An embedding is a numerical vector representation of data such as text. Similar meanings can be represented by vectors that are close to each other."
    },

    {
        "instruction": "What is self-attention?",
        "response": "Self-attention allows each token to consider other tokens in the sequence and determine which tokens are important for understanding the current token."
    },

    {
        "instruction": "What is a tokenizer?",
        "response": "A tokenizer converts text into tokens and then maps those tokens to numerical token IDs that a language model can process."
    },

    {
        "instruction": "What is an attention head?",
        "response": "An attention head is one attention mechanism inside multi-head attention. Different heads can learn different relationships between tokens."
    },

    {
        "instruction": "What is QLoRA?",
        "response": "QLoRA combines quantization with LoRA. The pretrained model is loaded in a low-bit representation while LoRA adapters are trained on top of it."
    },

    {
        "instruction": "What is a vector database?",
        "response": "A vector database stores vector embeddings and allows similarity searches to retrieve information that is semantically related to a query."
    },

    {
        "instruction": "What is hallucination in an LLM?",
        "response": "Hallucination occurs when a language model generates information that sounds plausible but is incorrect, unsupported, or fabricated."
    },

    {
        "instruction": "What is prompt engineering?",
        "response": "Prompt engineering is the process of designing instructions and context that guide a language model toward a desired response."
    },

    {
        "instruction": "What is cross entropy loss?",
        "response": "Cross entropy loss measures the difference between the probability distribution predicted by a classification or language model and the correct target distribution."
    },

    {
        "instruction": "What is gradient descent?",
        "response": "Gradient descent is an optimization algorithm that updates model parameters in a direction that reduces the training loss."
    },

    {
        "instruction": "What is backpropagation?",
        "response": "Backpropagation calculates how much each trainable parameter contributed to the loss by propagating gradients backward through the neural network."
    },

    {
        "instruction": "What is an epoch?",
        "response": "An epoch is one complete pass through the training dataset."
    },

    {
        "instruction": "What is inference?",
        "response": "Inference is the process of using a trained model to generate predictions or responses for new input data."
    },

    {
        "instruction": "What is supervised fine-tuning?",
        "response": "Supervised fine-tuning trains a pretrained model using examples containing an input and a desired output."
    }

]

CONVERT TO HUGGING FACR DATASET

In [15]:
dataset = Dataset.from_list(data)

print(dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 20
})


In [16]:
print(dataset[0])

{'instruction': 'What is LoRA?', 'response': 'LoRA stands for Low-Rank Adaptation. It is a parameter-efficient fine-tuning technique that freezes the pretrained model and trains small low-rank matrices.'}


TRAINING ,VALIDATION SPLIT

In [17]:
dataset_split = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(eval_dataset))

Training examples: 16
Validation examples: 4


In [18]:
def format_example(example):

    text = f"""### Instruction:
{example["instruction"]}

### Response:
{example["response"]}{tokenizer.eos_token}"""

    return {
        "text": text
    }

In [19]:
train_dataset = train_dataset.map(format_example)
eval_dataset = eval_dataset.map(format_example)

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [20]:
print(train_dataset[0]["text"])

### Instruction:
What is hallucination in an LLM?

### Response:
Hallucination occurs when a language model generates information that sounds plausible but is incorrect, unsupported, or fabricated.<|im_end|>


TOKENIZATION

In [21]:
sample_text = train_dataset[0]["text"]

tokens = tokenizer.tokenize(sample_text)

print(tokens)

['###', 'ĠInstruction', ':Ċ', 'What', 'Ġis', 'Ġhalluc', 'ination', 'Ġin', 'Ġan', 'ĠL', 'LM', '?ĊĊ', '###', 'ĠResponse', ':Ċ', 'Hall', 'uc', 'ination', 'Ġoccurs', 'Ġwhen', 'Ġa', 'Ġlanguage', 'Ġmodel', 'Ġgenerates', 'Ġinformation', 'Ġthat', 'Ġsounds', 'Ġplausible', 'Ġbut', 'Ġis', 'Ġincorrect', ',', 'Ġunsupported', ',', 'Ġor', 'Ġfabricated', '.', '<|im_end|>']


In [22]:
token_ids = tokenizer.encode(sample_text)

print(token_ids)

[14374, 29051, 510, 3838, 374, 58023, 2554, 304, 458, 444, 10994, 1939, 14374, 5949, 510, 71845, 1754, 2554, 13666, 979, 264, 4128, 1614, 26885, 1995, 429, 10362, 49334, 714, 374, 15114, 11, 40409, 11, 476, 69454, 13, 151645]


TOKENIZE DATASET

In [23]:
MAX_LENGTH = 256

def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [24]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=False,
    remove_columns=train_dataset.column_names
)

tokenized_eval = eval_dataset.map(
    tokenize_function,
    batched=False,
    remove_columns=eval_dataset.column_names
)

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [25]:
print(tokenized_train[0].keys())

dict_keys(['input_ids', 'attention_mask'])


DATA COLLECTOR

In [26]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

LORA CONFIGURATION

In [27]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

APPLY LORA

In [29]:
!pip install -U torchao peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 40.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [30]:
model = get_peft_model(
    model,
    lora_config
)

In [31]:
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [32]:
for name, param in model.named_parameters():

    if param.requires_grad:
        print(name)

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.k_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight
base_m

TRAINING CONFIGURATION

In [33]:
training_args = TrainingArguments(
    output_dir="./lora_output",

    num_train_epochs=5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=1,

    save_strategy="epoch",

    fp16=torch.cuda.is_available(),

    report_to="none",

    remove_unused_columns=False
)

In [34]:
per_device_train_batch_size=2

In [35]:
gradient_accumulation_steps=4

CREATE TRAINER

In [36]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_eval,

    data_collator=data_collator
)

LORA START

In [37]:
trainer.train()

Step,Training Loss
1,3.448772
2,3.393766
3,3.250158
4,3.007545
5,2.941527
6,2.802318
7,2.682394
8,2.740871
9,2.543534
10,2.701605


TrainOutput(global_step=10, training_loss=2.951249027252197, metrics={'train_runtime': 20.9359, 'train_samples_per_second': 3.821, 'train_steps_per_second': 0.478, 'total_flos': 7051792433664.0, 'train_loss': 2.951249027252197, 'epoch': 5.0})

EVALUATE

In [38]:
eval_results = trainer.evaluate()

print(eval_results)

Training Loss,Validation Loss,Step
2.701605,2.536923,10


{'eval_loss': 2.5369226932525635}


In [43]:
adapter_path = "./my_lora_adapter"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved to:", adapter_path)

LoRA adapter saved to: ./my_lora_adapter


In [44]:
import os

for file in os.listdir(adapter_path):
    print(file)

README.md
chat_template.jinja
tokenizer_config.json
tokenizer.json
adapter_model.safetensors
adapter_config.json


GENERATION

In [46]:
prompt = """### Instruction:
What is LoRA?

### Response:
"""

response = generate_response(
    model,
    tokenizer,
    prompt,
    max_new_tokens=100
)

print(response)

### Instruction:
What is LoRA?

### Response:
LoRA stands for Local Robustness with Adaptive Regularization. It is a variant of the pre-trained ResNet model that aims to improve its performance by leveraging local information and adaptively adjusting parameters based on the current layer's features. It was introduced in 2018 and has since become a popular choice for training deep learning models due to its ability to achieve state-of-the-art results on various tasks. LoRA combines several existing techniques, such as residual connections, shortcut connections, and drop


COMPARE BEFORE AND AFTER

In [47]:
test_prompts = [
    "What is LoRA?",
    "What is RAG?",
    "What is a Transformer?",
    "What is fine-tuning?",
    "What is QLoRA?"
]

In [48]:
for prompt in test_prompts:

    formatted_prompt = f"""### Instruction:
{prompt}

### Response:
"""

    print("=" * 70)
    print("QUESTION:", prompt)

    result = generate_response(
        model,
        tokenizer,
        formatted_prompt,
        max_new_tokens=100
    )

    print(result)

QUESTION: What is LoRA?
### Instruction:
What is LoRA?

### Response:
LoRA stands for Low-Rank Approximate Randomization. It's a technique that improves the performance of pre-trained models by making small changes to them, while still keeping their structure intact. Specifically, LoRA reduces the number of parameters in an model by replacing some of them with random values, which helps the model converge faster and more reliably. This approach is particularly useful when dealing with large-scale models or those with very deep architectures. For example, it can help reduce overfitting in deep learning
QUESTION: What is RAG?
### Instruction:
What is RAG?

### Response:
RAG stands for Researching, Analyzing and Gaining Knowledge. It is a process that involves collecting data on various aspects of something or someone and then analyzing it to find patterns, trends, and insights. This can help researchers better understand the context, identify patterns, and draw conclusions based on their

RELOAD BASE MODEL

In [49]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

base_model = base_model.to(device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

LOAD LORA ADAPTER

In [50]:
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

TEST,RELOADED MODEL

In [51]:
prompt = """### Instruction:
Explain what PEFT is.

### Response:
"""

response = generate_response(
    fine_tuned_model,
    tokenizer,
    prompt,
    max_new_tokens=100
)

print(response)

### Instruction:
Explain what PEFT is.

### Response:
PEFT stands for "Patient Education Fund". It is a nonprofit organization that provides free healthcare services to low-income individuals and families in the United States. Their mission is to ensure access to quality health care, particularly in underserved areas. They provide medical assistance, education programs, and other resources to help people improve their health and wellbeing. If you are in need of health care services or have questions about how you can support PEFT's efforts, please feel free to contact them directly. ### Instruction:


MERGE LORA WITH BASE MODEL

In [52]:
merged_model = fine_tuned_model.merge_and_unload()

SAVED

In [53]:
merged_path = "./merged_lora_model"

merged_model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

print("Merged model saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved.


                         DATASET
                            │
                            ▼
                    Instruction / Response
                            │
                            ▼
                       TOKENIZER
                            │
                            ▼
                        TOKEN IDs
                            │
                            ▼
                 ┌─────────────────────┐
                 │   PRETRAINED QWEN   │
                 │                     │
                 │   Frozen Weights    │
                 │         ❄️          │
                 └──────────┬──────────┘
                            │
                     Selected Layers
                            │
             ┌──────────────┼──────────────┐
             ▼              ▼              ▼
          q_proj          k_proj          v_proj
             │              │              │
             ▼              ▼              ▼
           LoRA            LoRA            LoRA
             │              │              │
             └──────────────┼──────────────┘
                            │
                            ▼
                         Output
                            │
                            ▼
                           Loss
                            │
                            ▼
                     Backpropagation
                            │
                            ▼
                     LoRA Gradients
                            │
                            ▼
                       AdamW Update
                            │
                            ▼
                    TRAINED ADAPTER